In [ ]:
# Cell 1: Title and Introduction (Markdown)
"""
# Diffusion Lens Analysis for Dreambooth Model

This notebook walks through the process of applying the Diffusion Lens technique
to analyze the intermediate steps of a diffusion model fine-tuned using Dreambooth.
It mirrors the logic in `analyze_dreambooth_lens.py`.

**Goal:** Visualize how the image generation process evolves at different stages
(controlled by `start_layer`, `end_layer`, `step_layer`) for a specific prompt,
using the fine-tuned Dreambooth model.
"""

In [ ]:
# Cell 2: Imports (Code)
import torch
import os
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler  # Using DDPMScheduler as per inference script
from transformers import CLIPTextModel, CLIPTokenizer
from PIL import Image
from collections import OrderedDict  # For loading state dicts
from IPython.display import display # To display images inline

# Import the custom Diffusion Lens pipeline
# Make sure DiffusionLens folder is in the right place relative to this notebook
try:
    from DiffusionLens.pipeline_stable_diffusion import StableDiffusionPipeline as StableDiffusionGlassPipeline
except ImportError as e:
    print(f"Error importing DiffusionLens pipeline: {e}")
    print("Please ensure the 'DiffusionLens' directory is in the same directory as this notebook or accessible in the Python path.")
    # You might need to add the project root to sys.path if running from a subdir
    # import sys
    # sys.path.append('..')
    # from DiffusionLens.pipeline_stable_diffusion import StableDiffusionPipeline as StableDiffusionGlassPipeline


# Import helpers from your 'src' directory
# Adjust path if necessary
try:
    from src.model_setup import load_unet
    # from src.utils import get_free_gpu # Optional: if you want dynamic GPU selection
except ImportError:
    print("Warning: Could not import from src. Ensure 'src' is in the Python path or adjust imports.")
    print("Using standard UNet loading as fallback.")
    # Define dummy function or raise error if essential
    def load_unet(base_model):
        print("Using standard from_pretrained method for UNet.")
        return UNet2DConditionModel.from_pretrained(base_model, subfolder="unet")
    # def get_free_gpu(): return 0 # Dummy for fallback

In [ ]:
# Cell 3: Configuration (Markdown)
"""
## Configuration

**<<< IMPORTANT: EDIT THESE VALUES >>>**

Set the paths to your Dreambooth model components and define the analysis parameters.
These correspond to the command-line arguments in `analyze_dreambooth_lens.py`.
"""

# Cell 4: Configuration Variables (Code)
# --- Model Paths ---
# Replace with your actual paths/IDs
MODEL_BASE = "stabilityai/stable-diffusion-2-1-base" # Base model used for Dreambooth training
UNET_PATH = "path/to/your/trained_unet.pt" # Path to your fine-tuned UNet weights file (.pt, .bin, .safetensors)
TEXT_ENCODER_PATH = "path/to/your/trained_text_encoder.pt" # Path to your fine-tuned text encoder weights file

# --- Analysis Parameters ---
PROMPT = "a photo of [V] dog playing fetch"  # Replace [V] with your Dreambooth trigger word
NEGATIVE_PROMPT = "low quality, blurry, deformed, noisy, text, words"
OUTPUT_DIR = "output/dreambooth_lens_notebook" # Directory to save output images
SEED = 42
NUM_IMAGES_PER_PROMPT = 1 # Keep this at 1 for simplicity in visualizing layers
NUM_INFERENCE_STEPS = 50 # Standard number of steps
GUIDANCE_SCALE = 7.5

# --- Diffusion Lens Parameters ---
# Control which intermediate steps (layers/timesteps) are visualized.
# The exact meaning depends on the pipeline implementation, but it generally
# slices the text encoder's hidden states or UNet's process.
START_LAYER = 0             # Start visualization from this step/layer index
END_LAYER = 12              # End visualization at this step/layer index (exclusive? check pipeline)
STEP_LAYER = 2              # Visualize every N steps/layers

# --- Device and Precision ---
# DEVICE = torch.device(f"cuda:{get_free_gpu()}" if torch.cuda.is_available() else "cpu") # Optional dynamic GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHT_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Configuration:")
print(f"  Model Base: {MODEL_BASE}")
print(f"  UNet Path: {UNET_PATH}")
print(f"  Text Encoder Path: {TEXT_ENCODER_PATH}")
print(f"  Prompt: '{PROMPT}'")
print(f"  Output Dir: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}, Weight Dtype: {WEIGHT_DTYPE}")
print(f"  Lens Params: Start={START_LAYER}, End={END_LAYER}, Step={STEP_LAYER}")

# Cell 5: Model Loading - Base Components (Markdown)
"""
## 1. Load Model Components

First, load the components from the base model that were *not* fine-tuned (or only partially):
- Tokenizer
- Scheduler (Using DDPMScheduler as per your `inference.py`)
- VAE (Variational Autoencoder)
"""

# Cell 6: Load Base Components (Code)
try:
    print("Loading base tokenizer, scheduler, vae...")
    tokenizer = CLIPTokenizer.from_pretrained(MODEL_BASE, subfolder="tokenizer")
    scheduler = DDPMScheduler.from_pretrained(MODEL_BASE, subfolder="scheduler")
    vae = AutoencoderKL.from_pretrained(MODEL_BASE, subfolder="vae", torch_dtype=WEIGHT_DTYPE)
    print("Base components loaded.")
except Exception as e:
    print(f"Error loading base components from {MODEL_BASE}: {e}")
    # Stop execution if base components fail
    raise

# Cell 7: Model Loading - Fine-tuned Structures (Markdown)
"""
Next, load the *structure* of the models that were fine-tuned:
- UNet (using `load_unet` from your `src` if available, otherwise standard `from_pretrained`)
- Text Encoder
"""

# Cell 8: Load Fine-tuned Structures (Code)
try:
    print("Loading UNet and Text Encoder structures...")
    # Load UNet structure (weights will be loaded next)
    unet = load_unet(MODEL_BASE)
    # Load Text Encoder structure (weights will be loaded next)
    text_encoder = CLIPTextModel.from_pretrained(MODEL_BASE, subfolder="text_encoder")
    print("Structures loaded.")
except Exception as e:
    print(f"Error loading UNet/Text Encoder structures from {MODEL_BASE}: {e}")
    raise

# Cell 9: Model Loading - Fine-tuned Weights (Markdown)
"""
Now, load the fine-tuned weights (`state_dict`) from your specified paths into the UNet and Text Encoder structures.
This includes handling the `module.` prefix often added during distributed training.
"""

# Cell 10: Load Fine-tuned Weights (Code)
try:
    # Load UNet weights
    print(f"Loading fine-tuned UNet weights from: {UNET_PATH}")
    unet_state_dict = torch.load(UNET_PATH, map_location="cpu")
    if not any("module." in k for k in unet_state_dict.keys()):
        unet.load_state_dict(unet_state_dict)
    else:
        print("  (Handling 'module.' prefix in UNet state_dict)")
        new_state_dict_unet = OrderedDict()
        for k, v in unet_state_dict.items():
            name = k[7:] if k.startswith("module.") else k
            new_state_dict_unet[name] = v
        unet.load_state_dict(new_state_dict_unet)
    print("UNet weights loaded.")

    # Load Text Encoder weights
    print(f"Loading fine-tuned Text Encoder weights from: {TEXT_ENCODER_PATH}")
    text_encoder_state_dict = torch.load(TEXT_ENCODER_PATH, map_location="cpu")
    if not any("module." in k for k in text_encoder_state_dict.keys()):
        text_encoder.load_state_dict(text_encoder_state_dict)
    else:
        print("  (Handling 'module.' prefix in Text Encoder state_dict)")
        new_state_dict_text_encoder = OrderedDict()
        for k, v in text_encoder_state_dict.items():
            name = k[7:] if k.startswith("module.") else k
            new_state_dict_text_encoder[name] = v
        text_encoder.load_state_dict(new_state_dict_text_encoder)
    print("Text Encoder weights loaded.")

except FileNotFoundError as e:
    print(f"Error: Could not find state dict file: {e}")
    print(f"Please ensure UNET_PATH ('{UNET_PATH}') and TEXT_ENCODER_PATH ('{TEXT_ENCODER_PATH}') are correct.")
    raise
except Exception as e:
    print(f"Error loading state dicts: {e}")
    raise

# Cell 11: Move to Device and Set Eval Mode (Markdown)
"""
Move all relevant model components to the target device (`cuda` or `cpu`) and set them to evaluation mode (`.eval()`), which disables dropout and other training-specific layers.
"""

# Cell 12: Move to Device and Eval (Code)
print(f"Moving models to {DEVICE} with dtype {WEIGHT_DTYPE}...")
unet.to(DEVICE, dtype=WEIGHT_DTYPE)
text_encoder.to(DEVICE, dtype=WEIGHT_DTYPE)
vae.to(DEVICE, dtype=WEIGHT_DTYPE)
unet.eval()
text_encoder.eval()
vae.eval()
print("Models moved and set to eval mode.")

# Cell 13: Instantiate Diffusion Lens Pipeline (Markdown)
"""
## 2. Instantiate Diffusion Lens Pipeline

Now, create an instance of the modified `StableDiffusionGlassPipeline` from the `DiffusionLens` code, passing in our loaded Dreambooth components.

**Note on Scheduler:** We loaded `DDPMScheduler` following `inference.py`. The Diffusion Lens pipeline was originally tested with `DPMSolverMultistepScheduler`. If you encounter errors during the pipeline run related to the scheduler, you might need to either:
    a) Load `DPMSolverMultistepScheduler.from_pretrained(MODEL_BASE, subfolder='scheduler')` and pass that instead.
    b) Modify the `StableDiffusionGlassPipeline` code if it strictly requires a specific scheduler type.
"""

# Cell 14: Instantiate Pipeline (Code)
print("Instantiating Diffusion Lens pipeline...")
try:
    pipe = StableDiffusionGlassPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet,
        scheduler=scheduler, # Using the DDPMScheduler loaded earlier
        safety_checker=None, # Disable safety checker for this analysis
        feature_extractor=None,
    )
    # pipe.to(DEVICE) # The pipeline class might not need .to(DEVICE) if components already are
    print("Diffusion Lens pipeline instantiated.")
except Exception as e:
    print(f"Error instantiating Diffusion Lens pipeline: {e}")
    print("Check compatibility, especially the scheduler type.")
    raise

# Cell 15: Prepare Output and Seed (Markdown)
"""
## 3. Prepare for Generation

- Create the output directory if it doesn't exist.
- Set the random seed for reproducibility.
"""

# Cell 16: Prepare Output and Seed (Code)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

generator = torch.Generator(DEVICE).manual_seed(SEED)
print(f"Using random seed: {SEED}")

# Cell 17: Run Pipeline (Markdown)
"""
## 4. Run the Pipeline

This is the core step where we call the Diffusion Lens pipeline. It will:
1. Encode the prompt using different layers of the text encoder (controlled by `start_layer`, `end_layer`, `step_layer`).
2. For *each* selected text encoder layer's output, run the full UNet denoising loop (`num_inference_steps`).
3. Decode the resulting latent representation using the VAE.

The pipeline is expected to return a *list* of outputs, where each item corresponds to the result obtained using a different text encoder layer's embedding.
"""

# Cell 18: Run Pipeline (Code)
print(f"Running pipeline for prompt: '{PROMPT}'")
print(f"Visualizing text encoder layers from {START_LAYER} to {END_LAYER} (exclusive?) with step {STEP_LAYER}")

try:
    # Use autocast for potential speedup with float16
    with torch.autocast(DEVICE, dtype=WEIGHT_DTYPE):
        layer_outputs = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
            num_images_per_prompt=NUM_IMAGES_PER_PROMPT,
            start_layer=START_LAYER,  # Pass lens parameters
            end_layer=END_LAYER,
            step_layer=STEP_LAYER,
            output_type="pil" # Get PIL images
        )
    print(f"Pipeline finished. Received {len(layer_outputs)} layer outputs.")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    print("This might be due to incompatible lens parameters (start/end/step_layer)")
    print("or issues with the custom pipeline implementation (e.g., scheduler).")
    raise

# Cell 19: Process and Save/Display Results (Markdown)
"""
## 5. Process and Save/Display Results

Iterate through the list of outputs returned by the pipeline. Each output corresponds to an image generated using a specific intermediate layer's embedding. Save each image and display it inline.
"""

# Cell 20: Process Results (Code)
print(f"Processing {len(layer_outputs)} results...")
current_layer_index = START_LAYER

for i, output in enumerate(layer_outputs):
    if not hasattr(output, 'images') or not output.images:
        print(f"Warning: No images found in output for layer index {i} (corresponds to approx layer {current_layer_index})")
        continue

    # Get the first image (since num_images_per_prompt = 1)
    img = output.images[0]

    # Define filename
    # The actual layer index from the text encoder used is start_layer + i * step_layer
    actual_layer_num = START_LAYER + i * STEP_LAYER
    layer_filename = f"layer_{actual_layer_num:03d}_step_{i:03d}.png"
    output_path = os.path.join(OUTPUT_DIR, layer_filename)

    # Save the image
    img.save(output_path)
    print(f"Saved image for layer {actual_layer_num} to {output_path}")

    # Display the image inline
    print(f"Displaying image generated using Text Encoder Layer {actual_layer_num}:")
    display(img)
    print("-" * 30)

    # Increment for the next loop iteration's layer name (though actual_layer_num is calculated directly)
    # current_layer_index += STEP_LAYER # Not strictly needed if using actual_layer_num

print("Analysis complete. Images saved in:", OUTPUT_DIR)

# Cell 21: Conclusion (Markdown)
"""
## Conclusion

The notebook executed the Diffusion Lens analysis pipeline using your fine-tuned Dreambooth model. The generated images, each corresponding to using a different intermediate representation from the text encoder, have been saved to the specified output directory and displayed above.

By comparing these images, you can gain insights into how different levels of text representation influence the final generated image within your Dreambooth model.
"""